In [17]:
import random

from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer

# 加载训练数据
mnli = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(100_000)).remove_columns("idx")

# 过滤后获得正例的 premise/hypothesis 数据, 过滤出 33803 条记录
mnli = mnli.filter(lambda x: True if x["label"] == 0 else False)

# 定义训练数据集，三元组
# anchor(mnli 的 premise), positive(mnli 的 hypothesis), negative(mnli["hypothesis"] 中随机选取)
train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = random.sample(mnli["hypothesis"], len(mnli)) # 打乱 mnli["hypothesis"] 将作为负例
for row, soft_negative in zip(mnli, soft_negatives):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
train_dataset = Dataset.from_dict(train_dataset)

# 选择基座模型
embedding_model = SentenceTransformer('bert-base-uncased', device="cuda")

# 定义损失函数
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# 定义评估器，使用语义文本相似度基准(Semantic Textual Similarity Benchmark, STSB)
# 这是一个由人工标注的句子对数据集，相似度分数在 1 ~ 5 之间
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]], # 值转换为 0~1 之间
    main_similarity="cosine"
)

# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)

# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

print(evaluator(embedding_model))
trainer.train()
print(evaluator(embedding_model))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3182.14it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'pearson_cosine': 0.5917194531209226, 'spearman_cosine': 0.5931742011707938}


Step,Training Loss
100,0.319276
200,0.099703
300,0.069546
400,0.066483
500,0.061056
600,0.065160
700,0.061453
800,0.053882
900,0.062987
1000,0.054405


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]


{'pearson_cosine': 0.8087996401658296, 'spearman_cosine': 0.8130279205098503}


In [16]:
evaluator(embedding_model)

{'pearson_cosine': 0.8126082864719841, 'spearman_cosine': 0.8147842455653254}